<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_03_target_definition/stage_03_target_definition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_03_target_definition**


## **Configuración del Entorno**


### 0.1. Acceso a Drive

In [8]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Importación de librerías


In [9]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

### 0.3. Definición de rutas

In [10]:
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [11]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/processed/mnq_intraday.parquet"))
OUT_PARQUET = Path(os.environ.get("OUT_PARQUET", "data/processed/mnq_intraday_labeled.parquet"))
OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/target_definitio_summary.json"))

In [12]:
# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
OUT_PARQUET = DRIVE_DIR / OUT_PARQUET
OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

## **1. Definición de parámetros de operación**

El objetivo de este proyecto es la obtención de un rédito económico mediante la operatoria sobre el índice MNQ.

El plan de operación propuesto se define bajo los siguientes parámetros:

- Instrumento: MNQ
- Frecuencia operativa: hasta 4 operaciones diarias
- Meta diaria: mínimo USD 200, con un objetivo ideal de hasta USD 500
- Valor por punto del MNQ: USD 2

Esto se traduce en los siguientes objetivos diarios expresados en puntos:

- USD 200 ⇒ 100 puntos diarios
- USD 500 ⇒ 250 puntos diarios

Considerando un máximo de 4 operaciones por día, el objetivo promedio por operación resulta:

- Objetivo mínimo: 25 puntos por trade (100 / 4)
- Objetivo ideal: 62,5 puntos por trade (250 / 4)

Bajo esta premisa, es necesario evaluar la relación entre los retornos logarítmicos y la variación en puntos del índice, así como la frecuencia con la que se producen movimientos de esta magnitud.

## **2. Evaluación de feature de entrada `close`**




### 2.1. Justificación de análisis

A partir de la definición de los parámetros de operación establecidos en el punto anterior, resulta necesario analizar la variable `close`, la cual representa el nivel del índice MNQ en cada instante temporal del dataset intradía.

La evaluación de esta variable constituye un paso previo e indispensable al análisis estadístico de los retornos, por los siguientes motivos:

---

**1. Relación entre objetivos económicos y nivel del índice**

Los objetivos definidos en el Punto 1 se expresan en puntos del índice (25 a 62,5 puntos por operación). Sin embargo, los retornos utilizados en el modelado se calcularán en forma logarítmica, según la expresión:

$$
    r_t = \ln\left( \frac{close_{t+h}}{close_t} \right)
$$

Esto implica que un mismo desplazamiento en puntos absolutos del índice no se traduce en un retorno constante, sino que depende directamente del valor de `close_t`. Por lo tanto, para poder relacionar correctamente los objetivos económicos definidos en puntos con los retornos logarítmicos, es imprescindible conocer el orden de magnitud y la variabilidad del nivel del índice.

---

**2. Justificación del uso de close como referencia de escala**

La variable `close` actúa como factor de escala entre el retorno logarítmico y la variación absoluta en puntos del MNQ. En términos prácticos, la conversión puede aproximarse como:

$$
\Delta \text{puntos} \approx r_t \times close_t
$$

En consecuencia:

- definir umbrales de retorno sin considerar `close` conduce a criterios arbitrarios

- analizar `close` permite establecer umbrales de retorno dinámicos, coherentes con distintos niveles del índice.

---

**3. Necesidad de una referencia común antes del análisis estadístico**

Dado que el dataset abarca múltiples períodos y regímenes de mercado, el nivel del índice MNQ presenta variaciones significativas a lo largo del tiempo. Por este motivo, antes de evaluar la frecuencia y distribución de los retornos, es metodológicamente correcto:

  1. Comprender la escala real del índice representada por `close`.
  2. Validar que los objetivos definidos en puntos sean razonables en todo el período analizado.
  3. Evitar sesgos derivados de asumir un nivel de precios constante.

---

**4 Rol de este paso dentro del pipeline**

La evaluación de la variable `close` cumple, dentro del pipeline, la función de:

- Conectar los objetivos económicos con las variables financieras del dataset.
- Establecer una base sólida para la posterior evaluación estadística de los retornos.
- Garantizar coherencia entre la formulación del problema y la realidad operativa.

---

En síntesis, el análisis de la feature `close` no persigue inicialmente fines estadísticos, sino que constituye un paso conceptual y metodológico orientado a asegurar que los objetivos económicos definidos sean correctamente traducidos al lenguaje de los retornos financieros. Solo a partir de esta validación resulta pertinente avanzar al análisis estadístico de los retornos y a la definición final de los targets de predicción.

### 2.2. Aplicación de análisis

#### 2.2.1. Carga de dataset `intraday_mnq`


In [13]:
def load_mnq_parquet():
    os.path.exists(IN_PARQUET)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(IN_PARQUET)
    return mnq_parquet

In [14]:
def add_column_date(df):
    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [15]:
def info_dataset(df):
  print("Información del dataset:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}")

In [16]:
mnq_intraday = load_mnq_parquet()
mnq_intraday = add_column_date(mnq_intraday)
mnq_intraday.head()


Archivo encontrado en disco. Cargando dataset local...


,date,open,high,low,close,volume
datetime,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,7
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,89
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,34
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,53
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,3


In [17]:
info_dataset(mnq_intraday)

Información del dataset:

	Cantidad de días: 1303
	Registros por día: 571
	Hora diaria de inicio 06:30
	Hora diaria de final 16:00
	Zona horaria: America/New_York


#### 2.2.2. Análisis estadistico de `close`


In [18]:
import pandas as pd
from typing import Dict, Tuple

def analyze_close_statistics(
    df: pd.DataFrame,
    close_col: str = "close",
    percentiles=(0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99),
    by_year: bool = True
) -> Tuple[pd.Series, Dict[str, float], pd.DataFrame | None]:
    """
    Calcula estadísticas descriptivas de la variable close.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con índice datetime y columna close.
    close_col : str
        Nombre de la columna de precios (default: 'close').
    percentiles : tuple
        Percentiles a calcular.
    by_year : bool
        Si True, devuelve estadísticas agregadas por año.

    Retorna
    -------
    close_describe : pd.Series
        Resultado de df[close].describe(percentiles=...)
    summary : dict
        Resumen compacto con métricas clave.
    close_by_year : pd.DataFrame | None
        Estadísticas por año (o None si by_year=False).
    """

    close_series = df[close_col].dropna()

    # 1) Estadísticas globales
    close_describe = close_series.describe(percentiles=percentiles)

    # 2) Resumen compacto
    summary = {
        "count": int(close_series.count()),
        "mean": float(close_series.mean()),
        "std": float(close_series.std()),
        "min": float(close_series.min()),
        "p01": float(close_series.quantile(0.01)),
        "p05": float(close_series.quantile(0.05)),
        "p50": float(close_series.quantile(0.50)),
        "p95": float(close_series.quantile(0.95)),
        "p99": float(close_series.quantile(0.99)),
        "max": float(close_series.max()),
    }

    # 3) Estadísticas por año (opcional)
    close_by_year = None
    if by_year:
        tmp = df[[close_col]].copy()
        tmp["year"] = tmp.index.year

        close_by_year = tmp.groupby("year")[close_col].agg(
            count="count",
            mean="mean",
            median="median",
            p05=lambda s: s.quantile(0.05),
            p95=lambda s: s.quantile(0.95),
            min="min",
            max="max",
        )

    return close_describe, summary, close_by_year


### 2.2. Evaluación de resultados

In [19]:
close_desc, close_summary, close_yearly = analyze_close_statistics(mnq_intraday)

print(close_desc)
#print(close_summary)
#print(close_yearly)


count    744013.000000
mean      14762.034275
std        3566.973264
min        6765.750000
1%         8099.780000
5%         9110.500000
25%       12065.250000
50%       14430.750000
75%       17513.000000
95%       21284.000000
99%       21908.720000
max       22317.250000
Name: close, dtype: float64


Algunas conclusiones del análisis de la variable `close`:


1. Nivel típico del índice MNQ

    La media de la variable `close` se ubica en 14 762 puntos, con una mediana cercana (14 431 puntos).
    Esto indica que, a lo largo de todo el período analizado, el MNQ ha operado la mayor parte del tiempo en torno al nivel de 15 000 puntos, validando dicho valor como referencia central de escala.

2. Amplio rango de valores y múltiples regímenes

    El rango observado va desde 6 766 hasta 22 317 puntos, lo que evidencia:

    - La presencia de múltiples regímenes de mercado
    - Una evolución estructural del índice a lo largo del tiempo
    - La no estacionariedad del nivel de precios.

    Este comportamiento descarta el uso de un único valor fijo de referencia para todo el período.

3. Concentración de observaciones en un rango operativo claro

    El 50 % central de los datos (P25–P75) se encuentra entre 12 065 y 17 513 puntos, mientras que el 90 % de las observaciones (P5–P95) se concentra aproximadamente entre 9 110 y 21 284 puntos.

    Esto define un rango operativo predominante, dentro del cual se desarrollan la mayoría de las oportunidades intradía.

4. Implicancia directa sobre la conversión puntos ↔ retornos

    Dado que el mismo desplazamiento en puntos genera retornos distintos según el nivel de `close`, la variabilidad observada implica que:

    - Un objetivo fijo en puntos (ej. 25 o 60 puntos) no corresponde a un retorno fijo.
    - Los umbrales de retorno deben ser dinámicos y dependientes de close_t.

    Esta conclusión es central para evitar sesgos de escala en la definición posterior de targets.

5. Consistencia con el objetivo económico del proyecto

    El rango y la media observados son coherentes con los objetivos definidos en el Punto 1, ya que:

    - Los niveles típicos del índice permiten que movimientos de 25 a 60 puntos representen retornos intradía realistas.

    -Dichos movimientos no corresponden a eventos extremos en la mayor parte del período analizado.

**Conclusión general**

El análisis de la variable close confirma que:

- El nivel del índice MNQ presenta una variabilidad significativa pero acotada,
- La media cercana a 15 000 puntos es representativa a nivel global,
- Cualquier definición de retornos objetivo debe considerar close como factor de escala dinámico.

Con estas conclusiones establecidas, el siguiente paso lógico es evaluar cómo se comportan los retornos logarítmicos en relación con estos niveles de precio, y con qué frecuencia permiten alcanzar los objetivos económicos definidos.

## **3. Evaluación de los retornos logarítmicos**


Una vez establecidos los parámetros económicos de la operatoria (Punto 1) y analizado el nivel y la variabilidad del índice MNQ a través de la variable `close` (Punto 2), corresponde evaluar el comportamiento estadístico de los retornos logarítmicos, con el objetivo de determinar si los movimientos necesarios para alcanzar los objetivos económicos definidos ocurren con una frecuencia razonable.

Para este análisis se consideran los retornos acumulados a distintos horizontes temporales (`ret_30`, `ret_60`, `ret_90` y `ret_120`), los cuales representan la variación relativa del precio del índice en ventanas de 30, 60, 90 y 120 minutos, respectivamente.

### **3.1. Justificación del uso de retornos**

El análisis del movimiento del índice MNQ no se realiza directamente sobre el precio (`close`), sino sobre sus retornos logarítmicos, debido a razones metodológicas y financieras bien establecidas.

En particular, los retornos logarítmicos:

- permiten comparar movimientos de precios en distintos niveles del índice,
- son aditivos en el tiempo, lo que facilita el análisis en ventanas temporales,
- presentan propiedades estadísticas más estables que los precios absolutos,
- constituyen la forma estándar de modelar variaciones relativas en finanzas cuantitativas.

Dado que el objetivo del proyecto es evaluar movimientos relativos del mercado en horizontes intradía, el retorno logarítmico resulta una representación más adecuada que la variación absoluta del precio.

### **3.2 Definición y cálculo de los retornos**

El retorno logarítmico se define como:

$$
r_t = \ln\left( \frac{close_{t+h}}{close_t} \right)
$$

donde:

- `close_t` es el valor del índice en el instante actual,
- `close_{t+h}` es el valor del índice luego de un horizonte temporal
`h`.


En este proyecto se consideran retornos acumulados en cuatro horizontes:

- `ret_30`: retorno a 30 minutos
- `ret_60`: retorno a 60 minutos
- `ret_90`: retorno a 90 minutos
- `ret_120`: retorno a 120 minutos

Estos retornos representan la variación relativa del índice en ventanas temporales alineadas con la operatoria intradía planteada en el Punto 1.

#### **3.2.1. Cálculo de retornos por horizonte temporal**

In [20]:
def add_log_return(df):
    df['ret_30'] = df.groupby('date')['close'].transform(
        lambda x: np.log(x.shift(-30)) - np.log(x)
    )

    df['ret_60'] = df.groupby('date')['close'].transform(
        lambda x: np.log(x.shift(-60)) - np.log(x)
    )

    df['ret_90'] = df.groupby('date')['close'].transform(
        lambda x: np.log(x.shift(-90)) - np.log(x)
    )

    df['ret_120'] = df.groupby('date')['close'].transform(
        lambda x: np.log(x.shift(-120)) - np.log(x)
    )

    return df

In [21]:
mnq_intraday_with_returns = add_log_return(mnq_intraday)

In [22]:
mnq_intraday_with_returns.head()

,date,open,high,low,close,volume,ret_30,ret_60,ret_90,ret_120
datetime,,,,,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,7,0.000029,0.001031,0.000687,0.001059
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,89,0.000057,0.001059,0.000859,0.001145
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,34,0.000315,0.001117,0.000916,0.001231
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,53,0.000258,0.000974,0.000916,0.001231
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,3,0.000315,0.000916,0.000888,0.001346


### **3.3. Análisis descriptivo de retornos logarítmicos por horizonte temporal**

#### **3.3.1. Código de aplicación**

In [23]:
returns = ['ret_30', 'ret_60', 'ret_90', 'ret_120']

In [24]:
returns_stats = (
    mnq_intraday_with_returns[returns]
    .dropna()
    .describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
)

#### **3.3.2. Análisis y Conclusiones**

In [25]:
returns_stats

,ret_30,ret_60,ret_90,ret_120
count,587653.000000,587653.000000,587653.000000,587653.000000
mean,0.000016,0.000039,0.000071,0.000100
std,0.003010,0.004274,0.005276,0.006141
min,-0.053377,-0.056549,-0.049889,-0.057965
1%,-0.008783,-0.012448,-0.015042,-0.017120
5%,-0.004639,-0.006776,-0.008475,-0.009999
50%,0.000098,0.000185,0.000243,0.000330
95%,0.004292,0.006151,0.007751,0.009118
99%,0.008058,0.011347,0.013780,0.015915
max,0.082122,0.079896,0.083184,0.090994


1. Coherencia estadística entre horizontes

    Se observa un comportamiento consistente y esperable:
    - La media del retorno aumenta al ampliar el horizonte temporal.
    - El desvío estándar crece de forma monótona (30 → 120 min).

    Esto confirma que:
    - los retornos están correctamente calculados,
    - ventanas más largas capturan mayor acumulación de movimiento.

2. Media cercana a cero (propiedad intradía)

    En todos los horizontes, la media del retorno es muy próxima a cero:
    - no existe sesgo direccional sistemático,
    - el mercado intradía es esencialmente balanceado.

    Conclusión metodológica:
    - no es apropiado modelar retornos esperando una tendencia promedio,
    - el valor está en eventos específicos, no en el promedio.

3. Distribuciones aproximadamente simétricas

    Los percentiles negativos y positivos son de magnitud comparable:
    - colas negativas y positivas similares,
    - posibilidad real de movimientos alcistas y bajistas.

    Esto habilita:
    - modelos simétricos long/short,
    - formulaciones de clasificación direccional más adelante.

4) Incremento de magnitud con el horizonte

    Los percentiles altos crecen de forma clara:

    P95:
    - ret_30 ≈ 0.43 %
    - ret_60 ≈ 0.62 %
    - ret_90 ≈ 0.78 %
    - ret_120 ≈ 0.91 %

    P99:
    - ret_30 ≈ 0.81 %
    - ret_120 ≈ 1.59 %

    Conclusión:
    - los movimientos económicamente relevantes existen,
    - pero su frecuencia depende fuertemente del horizonte temporal.

5. Presencia de colas extremas

    Los valores mínimos y máximos muestran:
    - retornos extremos (±5 % a ±9 %),
    - asociados a eventos excepcionales (aperturas, noticias, shocks).

    Implicación:
    - estos eventos no deben usarse como referencia operativa base,
    - pero confirman que el dataset captura escenarios de estrés reales.

6. Rol de cada horizonte en el pipeline

    A partir del análisis (sin decidir targets aún):
    - 30 min: muy frecuente, pero movimientos pequeños.
    - 60 min: buen equilibrio entre frecuencia y amplitud.
    - 90–120 min: movimientos grandes, menor frecuencia, escenarios de continuidad.

    Esto permite, en el siguiente paso, alinear nuestros objetivos económicos con horizontes, sin forzar supuestos.



**Conclusión general**

El análisis descriptivo de los retornos logarítmicos muestra que:

- el mercado del MNQ presenta movimientos intradía de magnitud creciente con el horizonte,
- los retornos tienen propiedades estadísticas sanas y coherentes,
- existen movimientos suficientes para sustentar objetivos económicos razonables,
- pero no de forma uniforme ni constante.

Con estas conclusiones, el siguiente paso lógico es evaluar qué magnitudes de retorno son compatibles con los objetivos económicos definidos, y recién entonces definir los targets.

## **4. Vinculación entre nivel de precio (`close`), retornos y objetivos económicos**

### **4.1. Explicación metodológica**

El objetivo de este punto es integrar los resultados obtenidos en los Puntos 2 y 3 con los parámetros económicos definidos en el Punto 1, de modo de establecer una relación cuantitativa coherente entre:

- el nivel del índice MNQ (`close`),
- los retornos logarítmicos (`ret_**`) intradía,
- y los objetivos económicos expresados en puntos y dólares.

Esta vinculación es un paso intermedio indispensable antes de definir formalmente los targets de predicción.

**1. Principio de correspondencia entre puntos y retornos**

  Los objetivos económicos del proyecto se formulan en términos de variación absoluta del índice (puntos), mientras que el comportamiento estadístico del mercado se analiza mediante retornos logarítmicos. Para conectar ambos dominios se utiliza la relación:

  $$
    \Delta \text{puntos} \approx r_{t,h} \times close_t
  $$
      
  donde:

  - `𝑟_{𝑡,ℎ}` es el retorno logarítmico acumulado en un horizonte `ℎ`
  - `𝑐𝑙𝑜𝑠𝑒_𝑡` es el nivel del índice al inicio de la ventana.

  Esta expresión permite traducir cualquier retorno observado a una variación en puntos directamente interpretable desde el punto de vista operativo.
<br><br>
**2. Uso del nivel de precio como factor de escala dinámico**

Dado que el análisis de la variable `close` evidenció una variabilidad significativa del nivel del índice a lo largo del tiempo, la conversión entre retornos y puntos no puede basarse en un valor fijo de referencia.

Por el contrario:
- cada observación debe evaluarse en función de su propio `close_t`,
- los umbrales de retorno asociados a un objetivo en puntos se definen de forma dinámica y dependiente del nivel del mercado.

Este enfoque garantiza que:
- los criterios operativos sean coherentes en distintos regímenes de precio,
- los resultados no estén sesgados hacia períodos específicos del dataset.
<br><br>
**3. Integración de horizontes temporales y objetivos operativos**

La vinculación entre precio y retorno se realiza de manera conjunta con el horizonte temporal del movimiento, dado que:

- distintos horizontes presentan distintas combinaciones de frecuencia y magnitud,
- los objetivos económicos definidos requieren movimientos de cierta amplitud en ventanas temporales compatibles con la operatoria intradía.

En consecuencia, esta etapa no busca identificar un único horizonte óptimo, sino evaluar cómo cada horizonte contribuye al cumplimiento de los objetivos económicos, considerando tanto la magnitud del movimiento como su frecuencia histórica.
<br><br>
**4. Rol de esta vinculación en la definición de targets**

La finalidad de este punto no es aún fijar los targets, sino:

- determinar qué rangos de retorno son compatibles con los objetivos económicos definidos,
- identificar qué horizontes temporales concentran dichos movimientos,
- establecer una base cuantitativa objetiva para la definición posterior de los targets de predicción.
<br><br>
**Cierre del punto (explicativo)**

En síntesis, la vinculación entre el nivel de precio, los retornos logarítmicos y los objetivos económicos permite traducir los requerimientos operativos del proyecto al lenguaje estadístico del dataset, asegurando coherencia entre la formulación del problema, el comportamiento histórico del mercado y la realidad de la operatoria intradía.

### **4.2. Vinculación cuantitativa (desde el objetivo económico al retorno requerido)**




“Análisis cuantitativo top-down: desde el objetivo económico hacia los retornos requeridos.”

La vinculación cuantitativa se hace convirtiendo nuestros objetivos en puntos (definidos en el Punto 1) a umbrales de retorno logarítmico usando el nivel del índice (`close`) (Punto 2), y luego contrastándolos con la distribución de retornos (Punto 3).
<br><br>
**1. Conversión exacta (umbral dinámico)**

Para un objetivo de `Δ` puntos en un horizonte `ℎ`:
$$
r_{\text{umbral},t} \approx \ln\left(1 + \frac{\Delta}{close_t}\right) \approx \frac{\Delta}{close_t}
$$

En práctica intradía (movimientos pequeños), la aproximación `Δ/𝑐𝑙𝑜𝑠𝑒_𝑡` es suficiente y consistente con tus `ret_h`.
<br><br>
**2. Umbrales de retorno equivalentes usando percentiles de `close`**

Como close varía mucho (2019–2025), el retorno necesario para lograr los mismos puntos cambia. Tomamos 3 niveles representativos:

  - P05 close = 9 110.5
  - P50 close = 14 430.75
  - P95 close = 21 284.0

Objetivo mínimo: 25 puntos por trade

$$r_{25} \approx \frac{25}{close_t}$$

En P05: $$\frac{25}{9110.5} = 0.002744 \;\Rightarrow\; 0.274\%$$
En P50: $$\frac{25}{14430.75} = 0.001732 \;\Rightarrow\; 0.173\%$$
En P95: $$\frac{25}{21284} = 0.001174 \;\Rightarrow\; 0.117\%$$


Objetivo ideal: 62.5 puntos por trade

$$r_{62.5} \approx \frac{62.5}{close_t}$$
	​
En P05: $$\frac{62.5}{9110.5} = 0.006860 \;\Rightarrow\; 0.686\%$$
En P50: $$\frac{62.5}{14430.75} = 0.004331 \;\Rightarrow\; 0.433\%$$
En P95: $$ \frac{62.5}{21284} = 0.002936\;\Rightarrow\; 0.294\%$$



Conclusión cuantitativa inmediata:

- Lograr 25 pts requiere típicamente retornos del orden 0.12%–0.27% (según régimen de close).
- Lograr 62.5 pts requiere típicamente 0.29%–0.69%.

<br><br>
**3. Contraste con la distribución de retornos (ubicación de magnitudes)**

Los percentiles del lado positivo de la distribución de retornos muestran los siguientes valores:

- `ret_60`: P95 = 0.006151 (0.615%), P99 = 0.011347 (1.135%)
- `ret_90`: P95 = 0.007751 (0.775%), P99 = 0.013780 (1.378%)
- `ret_120`: P95 = 0.009118 (0.912%), P99 = 0.015915 (1.592%)

Lectura:

El umbral para 62.5 pts en el régimen bajo (≈0.686%) está cerca del P95 de ret_60 y por debajo del P95 de ret_90 → razonable, pero no “frecuente”.

El umbral para 25 pts (≈0.12%–0.27%) está muy por debajo de esos percentiles altos → debería ser mucho más frecuente.


### **4.3. Evaluación de la frecuencia empírica de movimientos económicamente relevantes (desde el objetivo hacia la frecuencia real)**





#### **4.3.1. Marco teórico**

La pregunta a responder ahora es: **¿es viable el objetivo planteado?**

Hasta este punto del análisis se ha establecido:

- Cuántos puntos por operación son necesarios para cumplir los objetivos operativos (Punto 1).
- En qué rangos de precio opera el índice a lo largo del tiempo (Punto 2).
- Cómo se distribuyen los retornos intradía para distintos horizontes temporales (Punto 3).

Sin embargo, aún resta responder una cuestión central: con qué frecuencia real el mercado alcanza dichos objetivos.

Este sub-punto tiene como finalidad responder a la siguiente pregunta clave:

**¿Con qué probabilidad histórica el MNQ se mueve al menos Δ puntos dentro de un horizonte intradía dado?**

---

**La frecuencia como criterio decisivo**

La existencia de un movimiento no implica necesariamente su viabilidad operativa. En particular, un desplazamiento del precio puede:

- Existir desde el punto de vista estadístico.
- Presentar una magnitud compatible con los objetivos planteados.
- Pero ocurrir con una frecuencia demasiado baja para sostener una operatoria diaria.

Por lo tanto:
- No es suficiente saber que un evento puede ocurrir.
- Es imprescindible conocer cada cuántas veces ocurre.

La frecuencia empírica es el factor que permite determinar:
- Si el objetivo mínimo es alcanzable de forma consistente.
- Si el objetivo ideal es realista o meramente excepcional.
- Cuántas operaciones diarias tienen sentido desde una perspectiva probabilística.

----

**Uso de un umbral dinámico de evaluación**

Dado que:
- Los objetivos operativos se definen en puntos absolutos (Δ).
- El índice opera en niveles de precio variables a lo largo del tiempo.

No resulta adecuado evaluar la condición mediante un umbral fijo del tipo:
`ret_h ≥ constante`

En su lugar, se utiliza un umbral dinámico, definido como:

$$ret_{t,h} \ge \frac{\Delta}{close_t}$$

donde:

- `Δ` representa el objetivo en puntos (por ejemplo, 25 o 62.5).
- `close_t` es el nivel del índice en el instante inicial `𝑡`.

Este enfoque garantiza que:
- La comparación sea homogénea en distintos regímenes de precio.
- No se sobreestimen ni subestimen oportunidades en función del nivel del índice.
- El análisis permanezca alineado con la lógica de la operatoria real.

---

Este análisis permite:

1. Cuantificar la viabilidad real de cada objetivo en cada horizonte.
2. Identificar qué horizontes temporales concentran más oportunidades.
3. Separar:
    - objetivos frecuentes y operables,
    - de objetivos eventuales o de extensión.
4. Fundamentar la futura definición de targets con evidencia empírica, no con supuestos.

En síntesis, este sub-punto transforma el análisis previo en una medida clave para la toma de decisiones: la frecuencia histórica de movimientos económicamente relevantes. Solo a partir de esta información es posible definir targets que sean estadísticamente defendibles y operativamente sostenibles.

#### **4.3.2. Cálculo de frecuencias**

Para cada horizonte $h \in \{30, 60, 90, 120\}$ y para cada objetivo expresado en puntos $\Delta$, se evalúa la siguiente condición:

$$
ret_{t,h} \ge \frac{\Delta}{close_t}
$$

La frecuencia empírica se define como el porcentaje de observaciones que cumplen dicha condición respecto del total de observaciones válidas.

Este análisis responde directamente a la pregunta:

**“¿En qué proporción de los casos el mercado se movió al menos $\Delta$ puntos en $h$ minutos?”**

In [26]:
import pandas as pd
import numpy as np

def compute_move_frequencies(
    df: pd.DataFrame,
    close_col: str = "close",
    return_cols = ("ret_30", "ret_60", "ret_90", "ret_120"),
    point_targets = (25, 62.5)
) -> pd.DataFrame:
    """
    Calcula la frecuencia empírica con la que el MNQ alcanza
    distintos objetivos en puntos para varios horizontes temporales.

    Parámetros
    ----------
    df : DataFrame
        Dataset con columna 'close' y retornos logarítmicos.
    close_col : str
        Nombre de la columna de precio.
    return_cols : tuple
        Columnas de retornos (ret_30, ret_60, etc.).
    point_targets : tuple
        Objetivos en puntos a evaluar (ej. 25, 62.5).

    Retorna
    -------
    DataFrame con frecuencias empíricas (%).
    """

    results = []

    for pts in point_targets:
        for ret_col in return_cols:
            mask = df[[close_col, ret_col]].dropna().index
            close_t = df.loc[mask, close_col]
            ret_t = df.loc[mask, ret_col]

            r_threshold = pts / close_t
            freq = (ret_t >= r_threshold).mean()

            results.append({
                "horizon": ret_col,
                "points_target": pts,
                "frequency": freq
            })

    return pd.DataFrame(results)

In [27]:
freq_df = compute_move_frequencies(
    mnq_intraday_with_returns,
    point_targets=(25, 62.5)
)

#### **4.3.3. Resultados y análisis**

In [28]:
freq_df

,horizon,points_target,frequency
0,ret_30,25.0,0.193837
1,ret_60,25.0,0.274931
2,ret_90,25.0,0.322976
3,ret_120,25.0,0.356314
4,ret_30,62.5,0.045582
5,ret_60,62.5,0.094440
6,ret_90,62.5,0.137008
7,ret_120,62.5,0.174630


**1. Comportamiento consistente con el horizonte temporal**

  Los resultados muestran un patrón monótono y coherente: para ambos objetivos en puntos (25 y 62.5), la frecuencia de cumplimiento aumenta sistemáticamente al ampliar el horizonte temporal de análisis.

  Esto confirma que:
  - la metodología de cálculo es correcta,
  - los retornos están correctamente alineados con el horizonte temporal,
  - ventanas más largas permiten capturar movimientos de mayor magnitud con mayor probabilidad.

<br>

**2. Viabilidad relativa del objetivo mínimo (25 puntos)**

El objetivo de 25 puntos presenta las siguientes frecuencias empíricas:
- 30 minutos: ~19 %
- 60 minutos: ~27 %
- 90 minutos: ~32 %
- 120 minutos: ~36 %

Estas cifras indican que:
- el movimiento ocurre con una frecuencia no despreciable,
- especialmente en horizontes de 60 a 90 minutos, donde se aproxima a 1 de cada 3 observaciones,
- el objetivo resulta estadísticamente viable, aunque no trivial.

Desde el punto de vista operativo, esto sugiere que el objetivo mínimo puede ser alcanzado de forma consistente bajo un esquema selectivo, no aleatorio.
<br><br>
**3. Baja frecuencia del objetivo ideal (62.5 puntos)**

El objetivo de 62.5 puntos muestra frecuencias sensiblemente menores:
- 30 minutos: ~4.6 %
- 60 minutos: ~9.4 %
- 90 minutos: ~13.7 %
- 120 minutos: ~17.5 %

Estos valores evidencian que:
- el movimiento existe, pero ocurre con baja frecuencia,
- incluso en horizontes extendidos, se mantiene por debajo del 20 %,
- corresponde a escenarios excepcionales, más asociados a impulsos direccionales fuertes.

Esto descarta su uso como objetivo base para una operatoria diaria sistemática.
<br><br>
**4. Diferenciación clara entre objetivo base y extensión**

El contraste entre ambos objetivos permite establecer una separación natural:
- 25 puntos → movimiento relativamente frecuente y operable.
- 62.5 puntos → movimiento de baja probabilidad, adecuado como extensión condicional.

Esta diferenciación emerge de los datos, no de supuestos externos.
<br><br>
**5. Implicancia directa sobre la planificación operativa**

Los resultados indican que:
- aumentar el horizonte temporal incrementa la probabilidad de alcanzar los objetivos,
- pero no de manera proporcional para objetivos ambiciosos.

En consecuencia:
- el incremento del horizonte no compensa completamente la baja frecuencia de movimientos extensos,
- la planificación debe priorizar objetivos alineados con la zona de mayor densidad probabilística.
<br><br>
**Conclusiones del Punto 4.3**

1. La frecuencia empírica es un criterio indispensable para evaluar la viabilidad real de los objetivos operativos.
2. El objetivo mínimo de 25 puntos es estadísticamente defendible, especialmente en horizontes de 60 a 90 minutos.
3. El objetivo ideal de 62.5 puntos presenta una probabilidad baja y no debe considerarse como expectativa base.
4. La diferencia de frecuencias justifica un enfoque operativo basado en:
    - objetivos frecuentes como núcleo,
    - extensiones ocasionales como complemento.
5. Estos resultados proporcionan una base objetiva para avanzar hacia la definición de targets, evitando decisiones arbitrarias.

### **4.4. Análisis empírico inverso del movimiento intradía en puntos**

#### **4.4.1. Marco conceptual**

##### **1. Motivación y propósito del análisis**

Hasta este punto del trabajo, la vinculación entre retornos, nivel de precio y objetivos económicos se ha abordado desde un enfoque top-down, partiendo de metas operativas previamente definidas (en puntos y dólares) y evaluando su viabilidad estadística a través de la frecuencia empírica de cumplimiento (Puntos 4.2 y 4.3).

Si bien dicho enfoque permite validar si un objetivo es razonable o no, aún persiste una cuestión fundamental:

**¿Qué magnitudes de movimiento intradía genera naturalmente el mercado, independientemente de los objetivos económicos propuestos?**

Con el fin de responder a esta pregunta y reforzar la solidez metodológica del análisis, se introduce en este punto un enfoque inverso o bottom-up, en el cual se deja que el comportamiento histórico del MNQ determine las magnitudes de movimiento más frecuentes.

##### **2. Enfoque metodológico**

El análisis empírico inverso consiste en estudiar directamente la magnitud efectiva de los movimientos intradía, expresada en puntos del índice, para distintos horizontes temporales.

Para ello, los retornos logarítmicos calculados previamente se transforman en variaciones absolutas en puntos mediante la relación:

  $$
    \Delta \text{puntos}_{t,h} \approx r_{t,h} \times close_t
  $$
      


Este procedimiento permite:
- abandonar temporalmente la referencia a objetivos prefijados,
- analizar el movimiento real del mercado en unidades directamente operativas,
- observar la distribución completa de los desplazamientos intradía.


##### **3. Qué se busca identificar**

A través de este análisis se busca:

- identificar la magnitud típica del movimiento intradía del MNQ para cada horizonte temporal,
- detectar rangos de puntos que concentran la mayor densidad probabilística,
- distinguir entre:
  - movimientos frecuentes y estructurales del mercado,
  - movimientos excepcionales asociados a colas de la distribución.

En particular, el foco no está puesto en los valores extremos, sino en aquellos desplazamientos que ocurren de manera recurrente y estable, y que por lo tanto constituyen una base más sólida para el diseño de una estrategia operativa sistemática.

##### **4. Rol de este análisis dentro del proceso de definición de targets**

El análisis empírico inverso no reemplaza a los objetivos económicos definidos inicialmente, sino que los complementa y valida desde la perspectiva del comportamiento real del mercado.

Su rol dentro del pipeline es:
- evitar la imposición de umbrales arbitrarios,
- permitir que los datos históricos del MNQ dicten las magnitudes de movimiento más frecuentes,
- proporcionar un criterio adicional, independiente y empírico, para la posterior definición de los targets de predicción.

##### **5. Cierre conceptual**

En síntesis, este sub-punto introduce una validación fundamental: los objetivos operativos no solo deben ser económicamente deseables y estadísticamente viables, sino también consistentes con la dinámica intradía real del mercado. Solo a partir de esta doble validación resulta posible avanzar hacia una definición de targets que sea robusta, defendible y alineada con una operatoria sostenible en el tiempo.

#### **4.4.2. Código de aplicación y cálculo**

In [31]:
import numpy as np
import pandas as pd

# ============================================================
# 4.4 — Análisis empírico inverso del movimiento intradía en puntos
# ============================================================
# Objetivo:
# 1) Convertir retornos logarítmicos (ret_h) a movimientos en puntos (Δpts_h)
# 2) Resumir la distribución de |Δpts_h| (magnitud) por horizonte
# 3) Calcular frecuencias del tipo P(|Δpts_h| >= umbral)
# 4) Estimar el "delta más frecuente" (modo aproximado) usando bins/histograma
#
# Nota clave:
# - "frecuencia >= umbral" responde a "al menos X puntos".
# - "modo" (más repetido) requiere histograma/bins, no umbrales.
# ============================================================


def compute_points_moves(
    df: pd.DataFrame,
    close_col: str = "close",
    return_cols=("ret_30", "ret_60", "ret_90", "ret_120"),
    prefix: str = "pts_",
    method: str = "approx",   # "approx" o "exact"
) -> pd.DataFrame:
    """
    Crea columnas de movimiento en puntos por horizonte:
      - approx:  Δpts ≈ ret_h * close_t
      - exact:   Δpts ≈ close_t * (exp(ret_h) - 1)

    ¿Cuál usar?
    - approx es muy buena para movimientos intradía pequeños.
    - exact es más fiel para colas (eventos grandes).
    """
    if method not in ("approx", "exact"):
        raise ValueError("method debe ser 'approx' o 'exact'.")

    out = df.copy()

    for rc in return_cols:
        h = rc.replace("ret_", "")  # "30", "60", "90", "120"
        if method == "exact":
            # Δpts = close_t * (e^{ret} - 1)
            out[f"{prefix}{h}"] = out[close_col] * (np.exp(out[rc]) - 1.0)
        else:
            # Aproximación lineal: Δpts ≈ ret * close
            out[f"{prefix}{h}"] = out[rc] * out[close_col]

    return out


def summarize_points_moves(
    df: pd.DataFrame,
    points_cols=("pts_30", "pts_60", "pts_90", "pts_120"),
    percentiles=(0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99),
    use_abs: bool = True
) -> pd.DataFrame:
    """
    Resume la distribución de movimientos en puntos por horizonte.

    use_abs=True  -> analiza magnitud |Δpts| (útil para objetivos económicos sin dirección).
    use_abs=False -> analiza Δpts con signo (útil si se quiere separar long/short).
    """
    data = df.loc[:, points_cols].copy()
    if use_abs:
        data = data.abs()

    summary = data.describe(percentiles=percentiles).T

    # Reorden de columnas para lectura
    cols_order = ["count", "mean", "std", "min"] + [f"{int(p*100)}%" for p in percentiles] + ["max"]
    summary = summary[[c for c in cols_order if c in summary.columns]]
    return summary


def points_threshold_frequencies(
    df: pd.DataFrame,
    points_cols=("pts_30", "pts_60", "pts_90", "pts_120"),
    thresholds=(10, 15, 20, 25, 30, 40, 50, 62.5, 75, 100),
    use_abs: bool = True
) -> pd.DataFrame:
    """
    Calcula frecuencias empíricas del tipo:
        P(|Δpts_h| >= umbral)

    IMPORTANTE:
    - Esto responde a "¿con qué frecuencia se mueve AL MENOS X puntos?".
    - NO responde a "¿cuál es el delta más repetido?" (eso se hace con histograma/modo).
    """
    data = df.loc[:, points_cols].copy()
    if use_abs:
        data = data.abs()

    rows = []
    for col in points_cols:
        s = data[col].dropna()
        for thr in thresholds:
            rows.append({
                "horizon_pts": col,
                "points_threshold": float(thr),
                "frequency": float((s >= thr).mean()),
                "n": int(s.shape[0])
            })

    return pd.DataFrame(rows)


def mode_points_by_bins(
    df: pd.DataFrame,
    points_cols=("pts_30", "pts_60", "pts_90", "pts_120"),
    bin_width: float = 5.0,
    max_abs_points: float = 300.0,
    use_abs: bool = True
) -> pd.DataFrame:
    """
    Estima el "delta más frecuente" (modo aproximado) usando histograma por bins.

    - bin_width: ancho del bin en puntos (ej. 5 puntos).
    - max_abs_points: recorta a [0, max_abs_points] (o [-max, +max si use_abs=False)
      para evitar que colas extremas dominen la escala del histograma.

    Retorna para cada horizonte:
    - bin_left, bin_right: límites del bin modal
    - mode_midpoint: punto medio del bin modal (estimación del modo)
    - bin_frequency: frecuencia del bin modal (aprox densidad)
    - n_used: cantidad de observaciones consideradas
    """
    rows = []
    for col in points_cols:
        s = df[col].dropna()

        # Usualmente interesa magnitud (sin dirección)
        if use_abs:
            s = s.abs()
            s = s[(s >= 0) & (s <= max_abs_points)]
            bins = np.arange(0, max_abs_points + bin_width, bin_width)
        else:
            s = s[(s >= -max_abs_points) & (s <= max_abs_points)]
            bins = np.arange(-max_abs_points, max_abs_points + bin_width, bin_width)

        if s.empty:
            rows.append({
                "horizon_pts": col,
                "bin_left": np.nan,
                "bin_right": np.nan,
                "mode_midpoint": np.nan,
                "bin_frequency": np.nan,
                "n_used": 0
            })
            continue

        # Histograma: counts por bin
        counts, edges = np.histogram(s.values, bins=bins)
        idx = int(np.argmax(counts))

        left = float(edges[idx])
        right = float(edges[idx + 1])
        midpoint = (left + right) / 2.0
        freq = float(counts[idx] / counts.sum())

        rows.append({
            "horizon_pts": col,
            "bin_left": left,
            "bin_right": right,
            "mode_midpoint": midpoint,
            "bin_frequency": freq,
            "n_used": int(s.shape[0])
        })

    return pd.DataFrame(rows)


In [47]:
# (A) Dataset con retornos: mnq_intraday_with_returns
#     - Debe contener: close y ret_30/ret_60/ret_90/ret_120

# 1) Convertir a puntos (elige método):
#    - "approx" (rápido, suficiente para intradía típico)
#    - "exact"  (más fiel para colas)
mnq_with_pts = compute_points_moves(
    mnq_intraday_with_returns,
    method="approx"  # o "exact"
)

# 2) Resumen por percentiles (magnitud en puntos)
pts_summary = summarize_points_moves(
    mnq_with_pts,
    points_cols=("pts_30", "pts_60", "pts_90", "pts_120"),
    use_abs=True
)
# 3) Frecuencias "al menos X puntos"
freq_pts = points_threshold_frequencies(
    mnq_with_pts,
    points_cols=("pts_30", "pts_60", "pts_90", "pts_120"),
    thresholds=(10, 15, 20, 25, 30, 40, 50, 62.5, 75, 100),
    use_abs=True
)
# 4) "Delta más frecuente" (modo aproximado) por bins
#    - bin_width=5 significa "modo por intervalos de 5 puntos"
modes = mode_points_by_bins(
    mnq_with_pts,
    points_cols=("pts_30", "pts_60", "pts_90", "pts_120"),
    bin_width=5.0,
    max_abs_points=300.0,
    use_abs=True
)

#### **4.4.3. Distribución empírica de |Δpuntos| por horizonte temporal**

Este primer resultado presenta la distribución de la magnitud absoluta del movimiento intradía, expresada en puntos del índice MNQ, para distintos horizontes temporales (30, 60, 90 y 120 minutos).

In [48]:
print("=== Distribución |Δpts| por horizonte ===\n")
pts_summary

=== Distribución |Δpts| por horizonte ===



,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
pts_30,704923.0,27.445000,31.298248,0.0,0.250002,1.500070,7.997591,18.019786,35.950696,84.686966,143.620227,1381.681382
pts_60,665833.0,40.001235,44.317640,0.0,0.499992,2.249885,11.753815,26.782521,52.894542,121.492043,204.367265,1399.090763
pts_90,626743.0,50.299615,54.376302,0.0,0.500009,2.750274,14.988925,34.212038,67.659027,150.406684,249.487806,1460.053659
pts_120,587653.0,59.889531,62.883501,0.0,0.749981,3.500311,18.237892,41.544095,81.337567,175.428589,284.574905,1576.307287


**1. Comportamiento general de la distribución**

Se observa un patrón consistente y esperable en todos los horizontes:
- El movimiento promedio (mean) aumenta de manera monótona con el horizonte temporal.
- El desvío estándar crece de forma proporcional, reflejando mayor dispersión a medida que se amplía la ventana temporal.
- La distribución presenta colas largas, evidenciadas por valores máximos muy elevados.

Este comportamiento confirma que:
- el procedimiento de conversión de retornos a puntos es correcto,
- el mercado acumula mayor desplazamiento cuanto mayor es el horizonte analizado.
<br><br>

**2. Movimiento típico del mercado (mediana)**

La mediana (50 %) representa el movimiento más representativo del mercado en cada horizonte:
- 30 min: ~18 puntos
- 60 min: ~27 puntos
- 90 min: ~34 puntos
- 120 min: ~42 puntos

Interpretación clave:

Estos valores describen el movimiento intradía típico del MNQ y constituyen una referencia empírica directa de la magnitud que el mercado genera de forma recurrente.
<br><br>

**3. Zona de alta densidad (rango intercuartílico)**

El rango P25–P75 define la región donde se concentra el 50 % central de las observaciones:
- 30 min: ~8 a ~36 puntos
- 60 min: ~12 a ~53 puntos
- 90 min: ~15 a ~68 puntos
- 120 min: ~18 a ~81 puntos

Esto indica que:
- en horizontes de 60 a 90 minutos, la mayor parte de los movimientos intradía se ubican entre 20 y 60 puntos,
- este rango coincide con la zona de mayor interés operativo desde el punto de vista económico.
<br><br>

**4. Movimientos grandes y colas de la distribución**

Los percentiles altos muestran la presencia de movimientos extensos:
- P95: entre ~85 y ~175 puntos, según el horizonte.
- P99: entre ~144 y ~285 puntos.

Estos valores corresponden a:
- eventos poco frecuentes,
- impulsos direccionales fuertes,
- situaciones de alta volatilidad.

Conclusión:

Estos movimientos no deben utilizarse como referencia para objetivos operativos base, pero confirman que el dataset captura correctamente escenarios extremos del mercado.
<br><br>

**5. Implicancias directas para la operatoria**

A partir de esta distribución se desprenden varias conclusiones clave:

1. El MNQ presenta un movimiento intradía típico bien definido, que crece con el horizonte temporal.
2. La zona de mayor densidad probabilística se encuentra en el rango de 20–40 puntos para horizontes de 60–90 minutos.
3. Movimientos superiores a 60 puntos pertenecen progresivamente a la cola de la distribución y deben considerarse como extensiones.

<br><br>
**Conclusión parcial del Punto 4.4.3**

El análisis de la distribución empírica de |Δpuntos| muestra que el mercado genera, de manera natural y recurrente, movimientos intradía de magnitud moderada, claramente identificables y estables en el tiempo. Estos resultados proporcionan una base objetiva y empírica para la posterior selección de targets alineados con la dinámica real del MNQ.

#### **4.4.4.Frecuencia empírica de movimientos en puntos por horizonte**

Este bloque evalúa la probabilidad histórica de que el MNQ registre un movimiento intradía cuya magnitud absoluta sea al menos un determinado umbral de puntos, para distintos horizontes temporales.

In [45]:
print("=== Frecuencias P(|Δpts| >= umbral) ===\n")
freq_pts.sort_values(["horizon_pts", "points_threshold"])

=== Frecuencias P(|Δpts| >= umbral) ===



,horizon_pts,points_threshold,frequency,n
30,pts_120,10.0,0.857962,587653
31,pts_120,15.0,0.791021,587653
32,pts_120,20.0,0.727329,587653
33,pts_120,25.0,0.668252,587653
34,pts_120,30.0,0.613148,587653
35,pts_120,40.0,0.513623,587653
36,pts_120,50.0,0.430601,587653
37,pts_120,62.5,0.345619,587653
38,pts_120,75.0,0.278984,587653
39,pts_120,100.0,0.182586,587653


**1. Patrón general de las frecuencias**

Se observa un comportamiento monótono y consistente en todos los horizontes:
- A medida que el umbral en puntos aumenta, la frecuencia de ocurrencia disminuye.
- Para un mismo umbral, la frecuencia crece con el horizonte temporal.

Este patrón valida:
- la coherencia del cálculo,
- la relación directa entre tiempo disponible y probabilidad de capturar movimientos de mayor magnitud.

<br>

**2. Umbrales de alta probabilidad (movimientos “casi seguros”)**

Considerando umbrales que ocurren en aproximadamente 70–80 % de los casos:
- 30 min: ≥ 10–15 puntos
- 60 min: ≥ 15–20 puntos
- 90 min: ≥ 15–20 puntos
- 120 min: ≥ 20–25 puntos

Interpretación:

Estos valores representan los movimientos intradía más frecuentes y estables, constituyendo el límite superior de lo que puede considerarse “alta probabilidad” en cada horizonte.

<br>

**3. Umbrales de probabilidad intermedia (zona operativa)**

Para frecuencias en el rango 30–60 %, se identifica una zona operativa natural:
- 30 min: ~20–30 puntos
- 60 min: ~25–40 puntos
- 90 min: ~25–40 puntos
- 120 min: ~30–50 puntos

Esta región coincide con:
- la mediana y el rango intercuartílico observados en la distribución de |Δpuntos|,
- los objetivos económicos mínimos planteados previamente.

<br>

**4. Movimientos de baja frecuencia (extensiones)**

Umbrales con frecuencia inferior al 20 % corresponden a:
- 30 min: ≥ 50–62.5 puntos
- 60 min: ≥ 62.5–75 puntos
- 90 min: ≥ 75–100 puntos
- 120 min: ≥ 75–100 puntos

Conclusión:

Estos movimientos existen, pero pertenecen a la cola de la distribución y deben considerarse como escenarios de extensión, no como expectativa base.

<br>

**5. Comparación entre horizontes (lectura transversal)**

- 30 minutos: alta rotación, pero limitada capacidad de capturar movimientos grandes.
- 60 minutos: equilibrio óptimo entre frecuencia y magnitud.
- 90 minutos: mayor probabilidad de capturar extensiones con una frecuencia aún razonable.
- 120 minutos: máxima probabilidad para movimientos grandes, a costa de menor flexibilidad intradía.

<br>

**6. Implicancias para la definición de objetivos**

Este análisis permite establecer criterios claros:

1. Los objetivos con alta frecuencia garantizan consistencia pero menor ganancia por operación.
2. Los objetivos con frecuencia intermedia representan el mejor compromiso riesgo–retorno.
3. Los objetivos de baja frecuencia deben reservarse como extensiones condicionadas a contexto favorable.

<br>

**Conclusión parcial del Punto 4.4.4**

La frecuencia empírica de movimientos en puntos confirma que el MNQ presenta una estructura de desplazamientos intradía predecible en términos probabilísticos. Existen rangos de puntos claramente diferenciados según su frecuencia, lo que permite fundamentar la selección de objetivos operativos alineados con la dinámica real del mercado y el horizonte temporal elegido.

#### **4.4.5. Identificación del movimiento modal (delta más frecuente)**

Este resultado identifica el bin modal de la distribución de |Δpuntos|, es decir, el intervalo de magnitud que ocurre con mayor frecuencia para cada horizonte temporal.

In [46]:
print("=== Modo aproximado (bin modal) de |Δpts| ===\n")
modes.sort_values("horizon_pts")

=== Modo aproximado (bin modal) de |Δpts| ===



,horizon_pts,bin_left,bin_right,mode_midpoint,bin_frequency,n_used
3,pts_120,0.0,5.0,2.5,0.071970,582975
0,pts_30,0.0,5.0,2.5,0.161811,704507
1,pts_60,0.0,5.0,2.5,0.110763,664419
2,pts_90,0.0,5.0,2.5,0.088395,623812


**1. Resultado observado**

En todos los horizontes analizados, el bin modal corresponde a:
- Intervalo: 0–5 puntos
- Modo aproximado: 2.5 puntos

Frecuencia del bin modal:
- 30 min: ~16.2 %
- 60 min: ~11.1 %
- 90 min: ~8.8 %
- 120 min: ~7.2 %

<br>

**2. Interpretación correcta del resultado**

Este resultado **no indica** que el objetivo operativo deba ser 2.5 puntos.
Lo que indica es que: El movimiento intradía más frecuente del MNQ, medido minuto a minuto, es pequeño y corresponde a fluctuaciones de baja magnitud.

Esto es esperable en mercados financieros intradía:
- la mayor parte del tiempo el precio oscila dentro de rangos reducidos,
- los movimientos grandes se construyen por acumulación de micro-movimientos.

<br>

**3. Por qué el modo no es un buen target operativo**

Desde el punto de vista estadístico:
- el modo identifica el valor más frecuente,
- pero no necesariamente el más útil.

Desde el punto de vista operativo:
- movimientos de 0–5 puntos:
  - no cubren costos,
  - no compensan riesgo,
  - no permiten capturar el valor económico buscado.

Conclusión clave:
- El delta modal describe el “ruido” del mercado, no la oportunidad.

<br>

**4. Relación con los análisis anteriores**

Este resultado debe interpretarse en conjunto, no de forma aislada:

- Distribución (punto 4.4.3):
  - el movimiento típico (mediana) es mucho mayor (18–42 pts).

- Frecuencias por umbral (punto 4.4.4):
  - 20–40 pts ocurren con frecuencia relevante.

- Modo (punto 4.4.5):
  - 0–5 pts es el movimiento más común, pero de bajo valor económico.

Esto refuerza una idea central: **La estrategia no debe buscar el movimiento más frecuente, sino el movimiento frecuente y económicamente significativo.**

<br>

**5. Implicancia metodológica importante**

El análisis del modo cumple un rol clave:
- descarta explícitamente el uso del modo como criterio de target,
- evita una interpretación errónea del concepto de “lo que más se repite”,
- justifica por qué se utilizan percentiles y frecuencias acumuladas como criterios principales.

En otras palabras: **el modo explica qué hace el mercado la mayor parte del tiempo, pero los percentiles explican dónde está el valor operativo.**

<br>

**Conclusión parcial del Punto 4.4.5**

El movimiento modal del MNQ en todos los horizontes corresponde a oscilaciones de muy baja magnitud (0–5 puntos), asociadas al ruido intradía. Si bien este resultado es estadísticamente correcto, carece de relevancia económica directa. La identificación de este comportamiento refuerza la necesidad de definir objetivos operativos basados en movimientos menos frecuentes, pero suficientemente grandes como para justificar el riesgo asumido.

#### **4.4.6. Cierre de punto 4.4. (síntesis final)**

Combinando los tres bloques analizados:

1. El mercado presenta un movimiento típico de 20–40 puntos en 60–90 minutos.
2. Dichos movimientos ocurren con frecuencia suficiente para una operatoria sistemática.
3. El movimiento más frecuente (modo) es pequeño y no operable.

**Conclusión final:**

Los targets deben ubicarse por encima del ruido modal, dentro de la zona de alta densidad probabilística y relevancia económica, criterio que surge directamente del comportamiento empírico del mercado.

## **5. Definición formal de los targets de predicción**

### **5.1. Marco conceptual**

**1. Principio general de definición de targets**

La definición de los targets de predicción se fundamenta en la integración de tres criterios previamente establecidos:

1. Criterio económico (Punto 1): los movimientos deben ser monetariamente relevantes para la operatoria sobre MNQ.

2. Criterio estadístico–probabilístico (Puntos 3 y 4.3): los objetivos deben ocurrir con una frecuencia suficiente para sostener una operatoria sistemática.

3. Criterio empírico de mercado (Punto 4.4): los targets deben alinearse con las magnitudes de movimiento que el mercado genera de forma natural, evitando tanto el ruido intradía como las colas extremas.

Bajo este marco, el objetivo del modelo no es predecir el retorno exacto, sino identificar escenarios de movimiento intradía económicamente significativos dentro de un horizonte temporal determinado.

<br>

**2. Selección del horizonte temporal de predicción**

A partir del análisis conjunto realizado, se concluye que:

- horizontes cortos (30 min) presentan alta frecuencia pero baja magnitud,
- horizontes largos (120 min) capturan movimientos amplios con menor flexibilidad operativa,
- los horizontes 60 y 90 minutos ofrecen el mejor compromiso entre:
  - frecuencia empírica,
  - magnitud del movimiento,
  - compatibilidad con una operatoria intradía.

En consecuencia, el horizonte principal de predicción se define en 60 y 90 minutos, dejando otros horizontes como complementarios o experimentales.

<br>

**3. Definición del target principal (objetivo base)**

El análisis empírico muestra que:

- movimientos de 20 a 30 puntos:
  - representan el movimiento típico del mercado en 60–90 minutos,
  - ocurren con frecuencias del orden del 45–60 %,
  - superan claramente el ruido modal (0–5 puntos),
  - poseen relevancia económica directa.

Por lo tanto, se define como target principal la identificación de escenarios en los que el mercado se mueve **al menos 25 puntos** dentro del horizonte de predicción seleccionado.

Este target constituye el **núcleo operativo** del modelo.

<br>

**4. Definición del target de extensión (objetivo secundario)**

Movimientos de 50–65 puntos:
- se ubican en la parte alta de la distribución,
- presentan frecuencias del orden del 10–25 %, según el horizonte,
- corresponden a escenarios de impulso o continuidad direccional.

En consecuencia, estos movimientos se definen como targets de extensión, cuyo objetivo no es la ocurrencia diaria, sino la maximización del retorno en contextos favorables.

<br>

**5. Formulación del problema de predicción**

Con base en lo anterior, el problema de predicción se formula como:

- Problema de clasificación, no de regresión.
- El modelo debe estimar la probabilidad de que el movimiento intradía supere un umbral definido (ej. ≥ 25 puntos) dentro de un horizonte temporal dado.

Esta formulación:
- reduce la sensibilidad al ruido,
- alinea el output del modelo con decisiones operativas concretas,
- permite una interpretación probabilística directa.

<br>

**6. Rol de los targets dentro del pipeline**

Los targets definidos en este punto:
- constituyen la base del dataset labeled,
- se utilizarán en las etapas posteriores de:
  - entrenamiento,
  - validación,
  - comparación de modelos,
- podrán ajustarse dinámicamente según régimen de mercado, manteniendo el mismo marco conceptual.

<br>

**Cierre de marco conceptual**

La definición de los targets de predicción se apoya en una combinación equilibrada de criterios económicos, estadísticos y empíricos. Este enfoque evita tanto la arbitrariedad como el sobreajuste a eventos extremos, permitiendo construir modelos alineados con la dinámica real del MNQ y con una operatoria intradía sostenible.

### **5.2. Definición formal de etiquetas (labels) e implementación**